# context-renderer — Generate Context Sheets from MSSQL

**Workflow:**
1. Connect to your MSSQL database
2. Inspect the schema (tables, columns, PKs, FKs, indexes, routines)
3. Compute data metrics (null rates, cardinality, top values, min/max/avg)
4. Render and save Markdown context sheets

---

In [ ]:
from context_renderer import (
    MSSQLConnector,
    DatabaseInspector,
    DataMetrics,
    ContextSheetRenderer,
)

## 1 · Connect

**Option A** — fill in credentials directly:

In [ ]:
connector = MSSQLConnector(
    host="your-server",        # e.g. localhost, 192.168.1.10, myserver.database.windows.net
    database="your-database",
    username="sa",
    password="your-password",
    port=1433,
    trust_server_certificate=True,  # set False for production with valid cert
)

# Option B — read from .env file (see .env.example)
# connector = MSSQLConnector.from_env()

print("Connection OK:", connector.test_connection())

## 2 · Inspect the schema

In [ ]:
engine = connector.get_engine()

inspector = DatabaseInspector(engine)
db_schema = inspector.inspect(
    # schemas=["dbo"]  # Optional: restrict to specific schemas
)

tables = [t for t in db_schema.tables if t.table_type == "TABLE"]
views  = [t for t in db_schema.tables if t.table_type == "VIEW"]

print(f"  Database : {db_schema.database_name}")
print(f"  Schemas  : {db_schema.schemas}")
print(f"  Tables   : {len(tables)}")
print(f"  Views    : {len(views)}")
print(f"  Routines : {len(db_schema.routines)}")

In [ ]:
# Preview table list
for t in tables[:15]:
    pk = ", ".join(t.primary_key_columns) or "—"
    print(f"  {t.full_name:<40} {len(t.columns):>3} cols   {t.row_count or 0:>10,} rows   PK: {pk}")

## 3 · Compute data metrics

> **Optional** — skip for large databases or when you only need structural metadata.

In [ ]:
metrics_engine = DataMetrics(engine, top_n=5)
all_metrics = metrics_engine.compute_all(db_schema.tables)

print(f"Metrics computed for {len(all_metrics)} tables")

In [ ]:
# Preview first table
if all_metrics:
    tm = all_metrics[0]
    print(f"{tm.schema}.{tm.table_name} — {tm.row_count:,} rows\n")
    for cm in tm.column_metrics:
        top = ", ".join(str(v) for v, _ in cm.top_values[:3])
        print(f"  {cm.column_name:<30} null={cm.null_rate:.1%}  distinct={cm.distinct_count:,}  top=[{top}]")

## 4 · Render and save context sheets

In [ ]:
renderer = ContextSheetRenderer(
    schema=db_schema,
    table_metrics=all_metrics,  # Pass None to skip metrics columns
)

output_dir = f"./output/{db_schema.database_name}"
written = renderer.save(output_dir=output_dir)

print(f"✅  {len(written)} sheets saved to {output_dir}/")
for f in written[:6]:
    print(f"    {f}")
if len(written) > 6:
    print(f"    ... and {len(written) - 6} more")

## 5 · Preview sheets inline

In [ ]:
from IPython.display import Markdown

sheets = renderer.render_all()
Markdown(sheets["00_database_overview.md"])

In [ ]:
# Show a specific table — replace with your schema__table name
sheet_name = list(sheets.keys())[1]   # first table after overview
print(f"Showing: {sheet_name}")
Markdown(sheets[sheet_name])

---

In [ ]:
connector.close()
print("Connection closed.")